# DATA 140 Hands-On Activity: Machine Learning Regression

**Topic**: Linear and Nonlinear Regression  
**Focus**: Predicting outcomes, polynomial features, and avoiding overfitting

In this notebook you will find **three public datasets**. Each section includes a short description of the dataset, exploratory questions to get you started, and regression tasks to complete.

---

## How to use this notebook

1. **Choose one dataset** that interests you most.
2. **Explore the data** to understand dataset and identify which features best predict the target. Scatter plots and correlations are good starting points, but feel free to use other approaches.
3. **Complete the regression task** with these 5 steps:
   - Fit a multiple linear regression model
   - Add polynomial features to capture nonlinear patterns
   - Test different polynomial degrees to find the optimal complexity
   - Compare all models to see which generalizes best
   - Visualize predictions and check residuals for bias
4. **Report your metrics and interpret them.** The insights are often more interesting than the numbers themselves.

---

## Your goal

By the end of this activity, you should have a better intuition for:

- How regression models learn patterns from data
- When simple linear models work well, and when they don't
- What happens when you make models too complex (overfitting) or too simple (underfitting)
- How to use train/test splits to check if your model will work on new data



# Setup (Run this!!)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
except ImportError as e:
    raise ImportError("This notebook needs ipywidgets. Install with: pip install ipywidgets") from e

plt.style.use("seaborn-v0_8-whitegrid")


# Dataset 1: Used Car Price Prediction 🚗

**The dataset**: Cars with specifications like engine size, horsepower, weight, and fuel efficiency. Your goal is to predict car prices.

---

### Exploratory Questions

1. What is the distribution of car prices?
2. Which features correlate most strongly with price?
3. Do the relationships between features and price look linear?
4. Which features will you use for your models?

---

### Regression Tasks

**Step 1: Multiple Linear Regression**
- Fit a linear regression model using the features you identified
- Report: Training R², Test R², Training MSE, Test MSE, coefficients
- What does each coefficient tell you?

**Step 2: Add Polynomial Features**
- Try polynomial degree 2
- Report: new Test R², new Test MSE
- Did polynomial terms help? Why or why not?

**Step 3: Test Different Degrees**
- Test polynomial degrees 1-10 and plot training vs test error
- Report: Optimal degree (lowest test error), when does overfitting start?
- At what point does your model memorize the training data instead of learning real patterns? What does this tell you about the bias-variance tradeoff?

**Step 4: Compare Your Models**
- Compare Linear, Polynomial (degree=2), and Polynomial (optimal degree)
- Which model would you deploy and why?

**Step 5: Check the Residuals**
- Visualize residuals for your best model (residual plot, histogram, actual vs predicted)
- Are residuals randomly scattered? Any systematic bias? What does this tell you about what your model is missing?

In [ ]:
# Load used car dataset
car_url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/CarPrice.csv"
car_df = pd.read_csv(car_url)

print(f"Dataset shape: {car_df.shape}")
print(f"\nFirst few rows:")
car_df.head()

Dataset shape: (205, 26)

First few rows:


,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [ ]:
# Interactive regression lab for this dataset

def run_interactive_regression_lab(
    data,
    target_col,
    default_features=None,
    dataset_name="Dataset",
    degree_max=10,
    default_plot_feature=None,
):
    numeric_cols = [
        c for c in data.select_dtypes(include=[np.number]).columns
        if c != target_col
    ]

    if len(numeric_cols) == 0:
        display(Markdown("❗No numeric feature columns were found."))
        return

    if default_features is None:
        default_features = numeric_cols[: min(3, len(numeric_cols))]

    feature_widget = widgets.SelectMultiple(
        options=numeric_cols,
        value=tuple(default_features),
        description="Features",
        rows=min(10, len(numeric_cols)),
        layout=widgets.Layout(width="480px"),
    )

    test_size_widget = widgets.FloatSlider(
        value=0.2,
        min=0.1,
        max=0.5,
        step=0.05,
        description="Test size",
        continuous_update=False,
    )

    random_seed_widget = widgets.IntSlider(
        value=42,
        min=0,
        max=200,
        step=1,
        description="Random seed",
        continuous_update=False,
    )

    degree_widget = widgets.IntSlider(
        value=2,
        min=1,
        max=degree_max,
        step=1,
        description="Poly degree",
        continuous_update=False,
    )

    degree_min_widget = widgets.IntSlider(
        value=1,
        min=1,
        max=max(2, degree_max - 1),
        step=1,
        description="Deg min",
        continuous_update=False,
    )

    degree_max_widget = widgets.IntSlider(
        value=degree_max,
        min=2,
        max=degree_max,
        step=1,
        description="Deg max",
        continuous_update=False,
    )

    plot_feature_widget = widgets.Dropdown(
        options=numeric_cols,
        value=default_plot_feature or default_features[0],
        description="Plot x",
    )

    scale_widget = widgets.Checkbox(value=True, description="Scale features")

    def _fit_metrics(model, X_train, X_test, y_train, y_test):
        model.fit(X_train, y_train)
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)
        return {
            "Train R²": r2_score(y_train, train_pred),
            "Test R²": r2_score(y_test, test_pred),
            "Train MSE": mean_squared_error(y_train, train_pred),
            "Test MSE": mean_squared_error(y_test, test_pred),
            "Train MAE": mean_absolute_error(y_train, train_pred),
            "Test MAE": mean_absolute_error(y_test, test_pred),
            "train_pred": train_pred,
            "test_pred": test_pred,
        }

    def render(selected_features, test_size, random_seed, poly_degree, degree_min, degree_max_sel, plot_feature, scale_features):
        if len(selected_features) == 0:
            display(Markdown("❗Select at least one feature."))
            return

        if degree_min > degree_max_sel:
            display(Markdown("❗`Deg min` must be less than or equal to `Deg max`."))
            return

        selected_features = list(selected_features)
        X = data[selected_features].copy()
        y = data[target_col].copy()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_seed
        )

        steps_linear = []
        if scale_features:
            steps_linear.append(("scale", StandardScaler()))
        steps_linear.append(("model", LinearRegression()))
        linear_model = Pipeline(steps_linear)

        steps_poly = [("poly", PolynomialFeatures(degree=poly_degree, include_bias=False))]
        if scale_features:
            steps_poly.append(("scale", StandardScaler()))
        steps_poly.append(("model", LinearRegression()))
        poly_model = Pipeline(steps_poly)

        linear_metrics = _fit_metrics(linear_model, X_train, X_test, y_train, y_test)
        poly_metrics = _fit_metrics(poly_model, X_train, X_test, y_train, y_test)

        degree_values = list(range(degree_min, degree_max_sel + 1))
        train_mse_curve, test_mse_curve = [], []
        for d in degree_values:
            steps = [("poly", PolynomialFeatures(degree=d, include_bias=False))]
            if scale_features:
                steps.append(("scale", StandardScaler()))
            steps.append(("model", LinearRegression()))
            model_d = Pipeline(steps)
            m = _fit_metrics(model_d, X_train, X_test, y_train, y_test)
            train_mse_curve.append(m["Train MSE"])
            test_mse_curve.append(m["Test MSE"])

        best_idx = int(np.argmin(test_mse_curve))
        best_degree = degree_values[best_idx]

        best_steps = [("poly", PolynomialFeatures(degree=best_degree, include_bias=False))]
        if scale_features:
            best_steps.append(("scale", StandardScaler()))
        best_steps.append(("model", LinearRegression()))
        best_model = Pipeline(best_steps)
        best_metrics = _fit_metrics(best_model, X_train, X_test, y_train, y_test)

        summary = pd.DataFrame(
            [
                {
                    "Model": "Linear (deg=1)",
                    "Train R²": linear_metrics["Train R²"],
                    "Test R²": linear_metrics["Test R²"],
                    "Train MSE": linear_metrics["Train MSE"],
                    "Test MSE": linear_metrics["Test MSE"],
                    "Train MAE": linear_metrics["Train MAE"],
                    "Test MAE": linear_metrics["Test MAE"],
                },
                {
                    "Model": f"Polynomial (deg={poly_degree})",
                    "Train R²": poly_metrics["Train R²"],
                    "Test R²": poly_metrics["Test R²"],
                    "Train MSE": poly_metrics["Train MSE"],
                    "Test MSE": poly_metrics["Test MSE"],
                    "Train MAE": poly_metrics["Train MAE"],
                    "Test MAE": poly_metrics["Test MAE"],
                },
                {
                    "Model": f"Polynomial (best deg={best_degree})",
                    "Train R²": best_metrics["Train R²"],
                    "Test R²": best_metrics["Test R²"],
                    "Train MSE": best_metrics["Train MSE"],
                    "Test MSE": best_metrics["Test MSE"],
                    "Train MAE": best_metrics["Train MAE"],
                    "Test MAE": best_metrics["Test MAE"],
                },
            ]
        )

        display(Markdown(
            f"### {dataset_name} — interactive results\n"
            f"**Target:** `{target_col}` | **Features:** `{selected_features}` | "
            f"**Test size:** `{test_size:.2f}` | **Seed:** `{random_seed}`"
        ))
        display(summary.style.format({
            "Train R²": "{:.4f}", "Test R²": "{:.4f}",
            "Train MSE": "{:.4f}", "Test MSE": "{:.4f}",
            "Train MAE": "{:.4f}", "Test MAE": "{:.4f}",
        }))
        display(Markdown(f"**Best degree in selected range:** `{best_degree}` (lowest test MSE)"))

        if plot_feature not in X.columns:
            plot_feature = X.columns[0]

        one_feature = [plot_feature]
        X1 = data[one_feature]
        X1_train, X1_test, y1_train, y1_test = train_test_split(
            X1, y, test_size=test_size, random_state=random_seed
        )

        vis_steps = [("poly", PolynomialFeatures(degree=best_degree, include_bias=False))]
        vis_steps.append(("model", LinearRegression()))
        vis_model = Pipeline(vis_steps)
        vis_model.fit(X1_train, y1_train)

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        for ax, X_part, y_part, title in [
            (axes[0, 0], X1_train, y1_train, "Train fit"),
            (axes[0, 1], X1_test, y1_test, "Test fit"),
        ]:
            x_vals = X_part[plot_feature].to_numpy()
            order = np.argsort(x_vals)
            x_sorted = x_vals[order]
            y_hat = vis_model.predict(pd.DataFrame({plot_feature: x_sorted}))
            ax.scatter(x_vals, y_part, alpha=0.4, s=20)
            ax.plot(x_sorted, y_hat, color="black", linewidth=2)
            ax.set_title(title)
            ax.set_xlabel(plot_feature)
            ax.set_ylabel(target_col)

        axes[1, 0].plot(degree_values, train_mse_curve, marker="o", label="Train MSE")
        axes[1, 0].plot(degree_values, test_mse_curve, marker="o", label="Test MSE")
        axes[1, 0].axvline(best_degree, color="black", linestyle="--", alpha=0.7, label=f"Best degree = {best_degree}")
        axes[1, 0].set_title("Bias-variance view (MSE vs degree)")
        axes[1, 0].set_xlabel("Polynomial degree")
        axes[1, 0].set_ylabel("MSE")
        axes[1, 0].legend()

        residuals = y_test - best_metrics["test_pred"]
        sns.scatterplot(x=best_metrics["test_pred"], y=residuals, alpha=0.6, ax=axes[1, 1])
        axes[1, 1].axhline(0, color="red", linestyle="--")
        axes[1, 1].set_title("Residuals (test set)")
        axes[1, 1].set_xlabel("Predicted")
        axes[1, 1].set_ylabel("Residual (actual - predicted)")

        plt.tight_layout()
        plt.show()

    output = widgets.interactive_output(
        render,
        {
            "selected_features": feature_widget,
            "test_size": test_size_widget,
            "random_seed": random_seed_widget,
            "poly_degree": degree_widget,
            "degree_min": degree_min_widget,
            "degree_max_sel": degree_max_widget,
            "plot_feature": plot_feature_widget,
            "scale_features": scale_widget,
        },
    )

    display(
        widgets.VBox([
            feature_widget,
            widgets.HBox([test_size_widget, random_seed_widget, scale_widget]),
            widgets.HBox([degree_widget, degree_min_widget, degree_max_widget]),
            plot_feature_widget,
            output,
        ])
    )


In [ ]:
run_interactive_regression_lab(
    car_df,
    target_col="Price",
    default_features=["Mileage", "EngineV", "Year"],
    dataset_name="Used Car Price",
    degree_max=10,
    default_plot_feature="Mileage",
)


## Dataset 2: Heart Disease Prediction ❤️

**The dataset**: Patients with health metrics like age, blood pressure, cholesterol, and max heart rate. Your goal is to predict heart disease severity.

---

### Exploratory Questions

1. What is the distribution of heart disease severity?
2. Which features correlate most strongly with disease severity?
3. Do the relationships between features and disease severity look linear?
4. Which features will you use for your models?

---

### Regression Tasks

**Step 1: Multiple Linear Regression**
- Fit a linear regression model using the features you identified
- Report: Training R², Test R², Training MSE, Test MSE, and coefficients
- What does each coefficient tell you? (e.g., how much does disease severity increase with age?)

**Step 2: Add Polynomial Features**
- Try polynomial degree 2 to capture curves
- Report: New Test R², New Test MSE
- Did polynomial terms help? What new patterns you find?

**Step 3: Test Different Degrees**
- Test polynomial degrees 1-10 and plot training vs test error
- Report: Optimal degree (lowest test error), and when test error starts increasing
- What does this tell you about overfitting and the bias-variance tradeoff?

**Step 4: Compare Your Models**
- Compare Linear, Polynomial (degree=2), and Polynomial (optimal degree)
- Which model generalizes best to new data? Which would you trust for medical predictions and why?

**Step 5: Check the Residuals**
- Visualize residuals for your best model (residual plot, histogram, actual vs predicted)
- Are residuals randomly scattered around zero, or do you see patterns? Are there certain patient profiles where the model struggles?


In [ ]:
# Load heart disease dataset
heart_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
heart_columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
                 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
heart_df = pd.read_csv(heart_url, names=heart_columns, na_values="?")

# Drop rows with missing values
heart_df = heart_df.dropna()

print(f"Dataset shape: {heart_df.shape}")
print(f"\nFirst few rows:")
heart_df.head()

Dataset shape: (297, 14)

First few rows:


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [ ]:
run_interactive_regression_lab(
    heart_df,
    target_col="target",
    default_features=["age", "trestbps", "chol", "thalach", "oldpeak"],
    dataset_name="Heart Disease Severity",
    degree_max=10,
    default_plot_feature="age",
)


In [ ]:
# Tip: try small feature sets first, then add features and compare how Test MSE and Test R² change.


## Dataset 3: NCAA Basketball Performance 🏀

**The dataset**: College basketball teams with statistics like offensive efficiency, defensive efficiency, pace, shooting percentages, and rebounds. Your goal is to predict season wins.

---

### Exploratory Questions

1. What is the distribution of wins across teams?
2. Which features correlate most strongly with wins?
3. Do the relationships between features and wins look linear?
4. Which features will you use for your models?

---

### Regression Tasks

**Step 1: Multiple Linear Regression**
- Fit a linear regression model using the features you identified
- Report: Training R², Test R², Training MSE, Test MSE, and coefficients
- What does each coefficient tell you?

**Step 2: Add Polynomial Features**
- Try polynomial degree 2 to capture curves
- Report: New Test R², New Test MSE
- Did polynomial terms help? Does pace have a sweet spot (too fast = turnovers, too slow = fewer possessions)?

**Step 3: Test Different Degrees**
- Test polynomial degrees 1-10 and plot training vs test error
- Report: Optimal degree (lowest test error), and when test error starts increasing
- What does this tell you about overfitting and the bias-variance tradeoff?

**Step 4: Compare Your Models**
- Compare Linear, Polynomial (degree=2), and Polynomial (optimal degree)
- Which model generalizes best to new data? Which would you use to predict tournament success and why?

**Step 5: Check the Residuals**
- Visualize residuals for your best model (residual plot, histogram, actual vs predicted)
- Are residuals randomly scattered around zero, or do you see patterns? Are there certain team styles (e.g., very fast or very slow pace) where the model struggles?

In [ ]:
# Load NCAA basketball dataset
# Uncomment the line below if you haven't install kagglehub
# !pip install kagglehub

import kagglehub

basketball_path = kagglehub.dataset_download("andrewsundberg/college-basketball-dataset")
basketball_df = pd.read_csv(f"{basketball_path}/cbb.csv")

print(f"Dataset shape: {basketball_df.shape}")
print(f"\nFirst few rows:")
basketball_df.head()

Using Colab cache for faster access to the 'college-basketball-dataset' dataset.
Dataset shape: (4249, 24)

First few rows:


,TEAM,CONF,G,W,ADJOE,ADJDE,BARTHAG,EFG_O,EFG_D,TOR,...,FTRD,2P_O,2P_D,3P_O,3P_D,ADJ_T,WAB,POSTSEASON,SEED,YEAR
0,North Carolina,ACC,40,33,123.3,94.9,0.9531,52.6,48.1,15.4,...,30.4,53.9,44.6,32.7,36.2,71.7,8.6,2ND,1.0,2016
1,Wisconsin,B10,40,36,129.1,93.6,0.9758,54.8,47.7,12.4,...,22.4,54.8,44.7,36.5,37.5,59.3,11.3,2ND,1.0,2015
2,Michigan,B10,40,33,114.4,90.4,0.9375,53.9,47.7,14.0,...,30.0,54.7,46.8,35.2,33.2,65.9,6.9,2ND,3.0,2018
3,Texas Tech,B12,38,31,115.2,85.2,0.9696,53.5,43.0,17.7,...,36.6,52.8,41.9,36.5,29.7,67.5,7.0,2ND,3.0,2019
4,Gonzaga,WCC,39,37,117.8,86.3,0.9728,56.6,41.1,16.2,...,26.9,56.3,40.0,38.2,29.0,71.5,7.7,2ND,1.0,2017


In [ ]:
numeric_basketball_cols = basketball_df.select_dtypes(include=[np.number]).columns.tolist()
default_basketball_features = [c for c in ["ADJOE", "ADJDE", "EFG_O", "EFG_D", "ORB", "DRB", "PACE"] if c in numeric_basketball_cols]
if "W" in numeric_basketball_cols:
    run_interactive_regression_lab(
        basketball_df,
        target_col="W",
        default_features=default_basketball_features[:5] if default_basketball_features else numeric_basketball_cols[:5],
        dataset_name="NCAA Basketball Wins",
        degree_max=10,
        default_plot_feature="ADJOE" if "ADJOE" in basketball_df.columns else (default_basketball_features[0] if default_basketball_features else numeric_basketball_cols[0]),
    )
else:
    display(Markdown("❗Expected target column `W` was not found in the basketball dataset."))


In [ ]:
# Reflection prompt:
# Which degree gives the best test MSE for each dataset,
# and where do you start seeing overfitting (train error keeps falling while test error rises)?
